In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
import re
import pickle
from dotenv import load_dotenv

import torch
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
query = "Kırmızı ışık"
top_k = 5

### Dense Search

In [6]:
client = QdrantClient("localhost", port=6333)

In [7]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
embed_model_name = "intfloat/multilingual-e5-small"
embed_model = SentenceTransformer(embed_model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8433.27it/s]


In [8]:
query_emb = embed_model.encode([query])[0]
dense_results = client.query_points(
    collection_name="legal_docs", query=query_emb.tolist(), limit=top_k
).points
dense_results = [(str(r.payload["chunk_id"]), r.score) for r in dense_results]

### BM25 Search

In [9]:
# Clean the texts
def tokenize_tr(text: str) -> list[str]:
    text = text.replace("İ", "i").replace("I", "ı").lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)

    return text.split()

In [10]:
with open("../data/processed/bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

In [11]:
bm25 = bm25_index["bm25"]
chunks = bm25_index["chunks"]

In [12]:
tokenized_query = tokenize_tr(query)
scores = bm25.get_scores(tokenized_query)
scored = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

sparse_results = [(chunks[idx]["chunk_id"], float(score)) for idx, score in scored]

## Hybrid Search

In [13]:
k = 60

In [15]:
rrf_scores = {}

# Dense sonuçlarını ekle
for rank, (chunk_id, _) in enumerate(dense_results):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank + 1)

# BM25 sonuçlarını ekle
# BM25 chunk dict döndürüyor, chunk_id'yi metadata'dan al
for rank, (chunk_id, _) in enumerate(sparse_results):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank + 1)

# Sırala
sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

# chunk_id → chunk dict mapping oluştur
id_to_chunk = {c["chunk_id"]: c for c in chunks}

In [16]:
hybrid_results = [
    {**id_to_chunk[cid], "rrf_score": rrf_scores[cid]}
    for cid in sorted_ids
    if cid in id_to_chunk
]

In [17]:
dense_results

[('trafik_kanunu_50', 0.84143734),
 ('trafik_kanunu_67', 0.8390462),
 ('trafik_kanunu_66', 0.83182836),
 ('trafik_kanunu_88', 0.8255878),
 ('trafik_kanunu_32', 0.81536835)]

In [18]:
sparse_results

[('trafik_kanunu_50', 5.6634736595001),
 ('trafik_kanunu_66', 5.190121807877882),
 ('trafik_kanunu_88', 4.299784153210367),
 ('trafik_kanunu_67', 2.7345179706023015),
 ('trafik_kanunu_120', 2.6002326659915327)]

In [23]:
rrf_scores

{'trafik_kanunu_50': 0.03278688524590164,
 'trafik_kanunu_67': 0.031754032258064516,
 'trafik_kanunu_66': 0.03200204813108039,
 'trafik_kanunu_88': 0.03149801587301587,
 'trafik_kanunu_32': 0.015384615384615385,
 'trafik_kanunu_120': 0.015384615384615385}

In [20]:
hybrid_results

[{'chunk_id': 'trafik_kanunu_50',
  'text': '47 – Karayollarından faydalananlar aşağıdaki sıralamaya göre;\na) Trafiği düzenleme ve denetimle görevli trafik zabıtası veya özel kıyafetli veya işaret\ntaşıyan diğer yetkili kişilerin uyarı ve işaretlerine,\nb) Trafik ışıklarına,\nc) Trafik işaret levhaları, cihazları ve yer işaretlemeleri ile belirtilen veya gösterilen\nhususlara,\nd) Trafik güvenliği ve düzeni ile ilgili olan ve yönetmelikte gösterilen diğer kural,\nyasak, zorunluluk veya yükümlülüklere,\nUymak zorundadırlar.\n(Değişik ikinci fıkra:12/2/2026-7574/11 md.) Bu maddenin birinci fıkrasının; (a)\nbendine uymayan sürücüler 3.000 Türk lirası, (b) bendine uymayan sürücüler 5.000 Türk lirası,\n(c) ve (d) bentlerine uymayan sürücüler 1.000 Türk lirası idari para cezası ile cezalandırılırlar.\n(Ek:18/10/2018-7148/20 md.) (Değişik üçüncü fıkra:12/2/2026-7574/11 md.) Işıklı\ntrafik işaretlerinden kırmızı renkli olanına uyma kuralının son ihlalin gerçekleştiği tarihten\ngeriye doğru bi